# Hospital Capacity Analysis – Data Cleaning

Author: David Saridakis  
Project: hospital-capacity-analysis  

## Description

This notebook performs the data cleaning and preparation process for the
Spanish hospital dataset obtained from the Ministerio de Sanidad.

The goal is to transform the raw dataset into a structured format suitable for storage in a relational database and further analysis.

Cleaning tasks include:

- Standardizing column names
- Handling missing values
- Correcting data types
- Normalizing categorical fields
- Validating numerical data
- Preparing the dataset for SQL ingestion

The cleaned dataset will be exported and loaded into a MySQL database to support SQL queries and dashboard visualizations in Looker Studio.

## Dataset Source

Catálogo Nacional de Hospitales – Spanish Ministry of Health

## Output

A cleaned dataset saved to:

data/cleaned/hospitals_clean.csv

This dataset will be used for database ingestion in the next stage of the project.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

In [2]:
# Load Dataset
df = pd.read_excel("../data/raw/hospitales_spain_raw.xlsx")
df.head()

,CCN,CODCNH,Nombre Centro,Dirección,Teléfono,Cód. Municipio,Municipio,Cód. Provincia,Provincia,Cód. CCAA,...,CAMAS,Cód. Clase de Centro,Clase de Centro,Cód. Dep. Funcional,Dependencia Funcional,Forma parte Complejo,CODIDCOM,Nombre del Complejo,ALTA,Email
0,1531000730,310121,Hospital García Orcoyen,Calle Santa Soria 22 -,848435000,310977,Estella-Lizarra,31,Navarra,15,...,108,C11,Hospitales Generales,2,Servicios e Institutos de Salud de Las Comunid...,N,NaN,NaN,N,gerencia.areasalud.estella@navarra.es
1,1531000738,310076,Clínica Arcángel San Miguel - Pamplona,Calle Beloso Alto 32 -,948296000,312016,Pamplona/Iruña,31,Navarra,15,...,72,C11,Hospitales Generales,20,Privados,N,NaN,NaN,N,clinicasanmiguel@clinicasanmiguel.es
2,1531000736,310057,Clínica Psiquiátrica Padre Menni,Calle Joaquín Beunza 45 -,948140611,312016,Pamplona/Iruña,31,Navarra,15,...,217,C14,Hospitales de salud mental y tratamiento de to...,20,Privados,N,NaN,NaN,N,gerencia.mennipamplona@hospitalarias.es
3,1531000733,310060,Clínica Universidad de Navarra,Avenida Pio Si 36 -,948255400,312016,Pamplona/Iruña,31,Navarra,15,...,230,C11,Hospitales Generales,20,Privados,N,NaN,NaN,N,direccioncun@unav.es
4,1531000737,310044,Hospital San Juan de Dios,Calle Beloso Alto 3 -,948231800,312016,Pamplona/Iruña,31,Navarra,15,...,288,C11,Hospitales Generales,20,Privados,N,NaN,NaN,N,hospitalpamplona@sjd.es


In [ ]:
# --- 1. Standardise Column Names ---

# Formatting (convert to snake_case)

df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("ó", "o")
    .str.replace("í", "i")
    .str.replace("á", "a")
    .str.replace("é", "e")
    .str.replace(".", "", regex=False)
)

# Check
df.columns


# Rename Key Columns (Spanish -> English)
df = df.rename(columns={
    "nombre_centro": "hospital_name",
    "direccion": "address",
    "telefono": "phone",
    "municipio": "municipality",
    "provincia": "province",
    "ccaa": "autonomous_community",
    "codigo_postal": "postcode",
    "camas": "beds",
    "dependencia_funcional": "management_type",
    "clase_de_centro": "center_type"
})

# Check
df.columns

Index(['ccn', 'codcnh', 'hospital_name', 'address', 'phone', 'cod_municipio',
       'municipality', 'cod_provincia', 'province', 'cod_ccaa',
       'autonomous_community', 'postcode', 'beds', 'cod_clase_de_centro',
       'center_type', 'cod_dep_funcional', 'management_type',
       'forma_parte_complejo', 'codidcom', 'nombre_del_complejo', 'alta',
       'email'],
      dtype='str')

In [11]:
# --- 2. Handle Missing Values ---

df.isna().sum()

# Most hopsitals are not part of a complex. We will 
# replace missing values with 'Independent'
df["nombre_del_complejo"] = df['nombre_del_complejo'].fillna('Independent')

# Email missing values replaced with 'Not Available'
df['email'] = df['email'].fillna('Not Available')

#Check
df.isna().sum()

ccn                       0
codcnh                    0
hospital_name             0
address                   0
phone                     0
cod_municipio             0
municipality              0
cod_provincia             0
province                  0
cod_ccaa                  0
autonomous_community      0
postcode                  0
beds                      0
cod_clase_de_centro       0
center_type               0
cod_dep_funcional         0
management_type           0
forma_parte_complejo      0
codidcom                724
nombre_del_complejo       0
alta                      0
email                     0
dtype: int64

In [ ]:
# --- 3. Fix Data Types ---

# Complex ID ('codidcom')
# converted to float due to missing values. Convert to 
# integer where possible.
df['codidcom'] = df['codidcom'].fillna(0).astype(int)


# Ensure 'beds' column is numeric.
df['beds'] = pd.to_numeric(df['beds'])

# Check for zero or negative beds
df[df['beds'] <= 0]

,ccn,codcnh,hospital_name,address,phone,cod_municipio,municipality,cod_provincia,province,cod_ccaa,...,beds,cod_clase_de_centro,center_type,cod_dep_funcional,management_type,forma_parte_complejo,codidcom,nombre_del_complejo,alta,email


In [ ]:
# --- 4. Remove Unnecessary Columns ---

# Codes not used for this analysis.
df = df.drop(columns=[
    "codcnh",
    "cod_municipio",
    "cod_provincia",
    "cod_ccaa"
])

In [ ]:
# --- 5. Check for Duplicate Records ---

df.duplicated().sum() # Returns zero, all good.

np.int64(0)

In [16]:
# --- 6. Final Validation ---

df.info() 
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   ccn                   840 non-null    int64
 1   hospital_name         840 non-null    str  
 2   address               840 non-null    str  
 3   phone                 840 non-null    str  
 4   municipality          840 non-null    str  
 5   province              840 non-null    str  
 6   autonomous_community  840 non-null    str  
 7   postcode              840 non-null    int64
 8   beds                  840 non-null    int64
 9   cod_clase_de_centro   840 non-null    str  
 10  center_type           840 non-null    str  
 11  cod_dep_funcional     840 non-null    int64
 12  management_type       840 non-null    str  
 13  forma_parte_complejo  840 non-null    str  
 14  codidcom              840 non-null    int64
 15  nombre_del_complejo   840 non-null    str  
 16  alta               

,ccn,hospital_name,address,phone,municipality,province,autonomous_community,postcode,beds,cod_clase_de_centro,center_type,cod_dep_funcional,management_type,forma_parte_complejo,codidcom,nombre_del_complejo,alta,email
0,1531000730,Hospital García Orcoyen,Calle Santa Soria 22 -,848435000,Estella-Lizarra,Navarra,C. Foral de Navarra,31200,108,C11,Hospitales Generales,2,Servicios e Institutos de Salud de Las Comunid...,N,0,Independent,N,gerencia.areasalud.estella@navarra.es
1,1531000738,Clínica Arcángel San Miguel - Pamplona,Calle Beloso Alto 32 -,948296000,Pamplona/Iruña,Navarra,C. Foral de Navarra,31006,72,C11,Hospitales Generales,20,Privados,N,0,Independent,N,clinicasanmiguel@clinicasanmiguel.es
2,1531000736,Clínica Psiquiátrica Padre Menni,Calle Joaquín Beunza 45 -,948140611,Pamplona/Iruña,Navarra,C. Foral de Navarra,31014,217,C14,Hospitales de salud mental y tratamiento de to...,20,Privados,N,0,Independent,N,gerencia.mennipamplona@hospitalarias.es
3,1531000733,Clínica Universidad de Navarra,Avenida Pio Si 36 -,948255400,Pamplona/Iruña,Navarra,C. Foral de Navarra,31008,230,C11,Hospitales Generales,20,Privados,N,0,Independent,N,direccioncun@unav.es
4,1531000737,Hospital San Juan de Dios,Calle Beloso Alto 3 -,948231800,Pamplona/Iruña,Navarra,C. Foral de Navarra,31006,288,C11,Hospitales Generales,20,Privados,N,0,Independent,N,hospitalpamplona@sjd.es


In [ ]:
# --- 7. Export Clean Dataset ---

df.to_csv("../data/cleaned/hospitals_clean.csv", index=False)

In [18]:
# --- 8. Save for SQL Import ---

df.to_csv("../data/cleaned/hospitals_clean_for_sql.csv", index=False)